# EHF Analysis Playground — Time-Series Momentum, interrogated

Run a classic **TSMOM** strategy on a diversified basket, then put it through the *honesty layer*: one-call tearsheet, a parallel parameter sweep, Deflated Sharpe / PBO / Monte-Carlo, a parameter-sensitivity heatmap, and a regime breakdown.

The punchline you're looking for: a backtest that looks good **alone** vs. what it looks like once you account for how many darts you threw. Kernel: **`.venv311`**.

In [ ]:
%matplotlib inline
import os, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)

import numpy as np, pandas as pd, matplotlib.pyplot as plt

from ehf.research.data.fmp_provider import FMPProvider
from ehf.research.data.schema import Timeframe
from ehf.research.data.quality import DataQualityWarning
from ehf.research.backtest.panel import Panel
from ehf.research.backtest.engine import run_backtest
from ehf.research.backtest.sweep import run_sweep
from ehf.research.config.spec import RunSpec, UniverseSpec, CostConfig
from ehf.research.results.registry import ExperimentRegistry
from ehf.research.analysis import runs as A, regime
from ehf.research.analysis.tearsheet import tearsheet, comparison_tearsheet

warnings.filterwarnings("always", category=DataQualityWarning)  # never hide data warnings
print("repo root:", ROOT)

## 1 — A diversified momentum basket
Cross-asset names (equity / bonds / gold / international) — TSMOM lives on diversification across trends.

In [ ]:
SYMBOLS = ["SPY", "QQQ", "IWM", "EFA", "EEM", "GLD", "SLV", "TLT", "HYG", "XLE"]
START, END = "2010-01-01", "2024-12-31"
bars = FMPProvider().get_bars(SYMBOLS, START, END, Timeframe.DAY)
print(f"{len(bars)} rows | {bars['symbol'].nunique()} symbols | "
      f"{bars['timestamp'].min().date()} -> {bars['timestamp'].max().date()}")

## 2 — Run TSMOM and log it
12-month lookback, vol-scaled. Costs + t+1-open fills on by default.

In [ ]:
reg = ExperimentRegistry()
spec = RunSpec(strategy="tsmom",
               params={"lookback": 252, "vol_window": 63, "target_vol": 0.12},
               universe=UniverseSpec(symbols=tuple(SYMBOLS)),
               start=START, end=END, costs=CostConfig(slippage_bps=1.0))
res = run_backtest(spec, bars)
h = reg.log_run(spec, res)
print("logged:", h)
pd.Series(res.metrics).round(4)

## 3 — One-call tearsheet
Equity, drawdown, rolling Sharpe, return distribution, performance table, robustness scorecard — all from the stored run.

In [ ]:
tearsheet(reg, h);  # returns a Figure (rendered inline)

## 4 — Parameter sweep (parallel, all cores)
Try a grid of momentum params. Each cell is logged and tagged with one `sweep_id`.

In [ ]:
grid = {"lookback": [63, 126, 189, 252], "vol_window": [21, 42, 63], "target_vol": [0.10, 0.15]}
base = RunSpec(strategy="tsmom", params={}, universe=UniverseSpec(symbols=tuple(SYMBOLS)),
               start=START, end=END, costs=CostConfig(slippage_bps=1.0))
sweep = run_sweep(base, grid, bars, registry=reg)   # 4*3*2 = 24 cells, parallel
sid = sweep["sweep_id"].iloc[0]
print(f"{len(sweep)} cells | sweep_id = {sid}")
sweep.sort_values("m_sharpe", ascending=False)[["config_hash","m_sharpe","m_cagr","m_max_drawdown"]].head(8)

## 5 — The honesty reveal: Deflated Sharpe & PBO
Raw Sharpe of the sweep's **best** cell vs. its score once we account for the 24 trials. This is the difference between a real edge and a lucky draw.

In [ ]:
best = sweep.sort_values("m_sharpe", ascending=False).iloc[0]
bh = best["config_hash"]
rep = A.robustness_report(reg, bh, sweep_id=sid, n_sims=1000)
mc = rep["monte_carlo"]
print(f"Best cell raw Sharpe        : {best['m_sharpe']:.2f}")
print(f"Probabilistic Sharpe (>0)   : {rep['psr_vs_zero']:.2f}")
print(f"DEFLATED Sharpe (N={rep['n_trials']} trials): {rep['deflated_sharpe']:.2f}   <-- the honest number")
print(f"PBO (prob. of overfitting)  : {rep['pbo']:.2f}")
print(f"Monte Carlo Sharpe [p5,med,p95]: [{mc['sharpe_p05']:.2f}, {mc['sharpe_median']:.2f}, {mc['sharpe_p95']:.2f}]")
print(f"Monte Carlo P(Sharpe>0)     : {mc['prob_sharpe_positive']:.0%} | worst drawdown: {mc['maxdd_worst']:.1%}")

## 6 — Parameter-sensitivity heatmap (plateau vs. needle)
Sharpe across `lookback` × `vol_window`. A broad bright region = robust; a lone bright cell = an overfit needle.

In [ ]:
piv = A.parameter_sensitivity(reg, sid, metric="sharpe", by=("lookback", "vol_window"))
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(piv.values, aspect="auto", cmap="viridis")
ax.set_xticks(range(len(piv.columns))); ax.set_xticklabels(piv.columns)
ax.set_yticks(range(len(piv.index))); ax.set_yticklabels(piv.index)
ax.set_xlabel("vol_window"); ax.set_ylabel("lookback"); ax.set_title("Sharpe by (lookback, vol_window)")
for i in range(piv.shape[0]):
    for j in range(piv.shape[1]):
        ax.text(j, i, f"{piv.values[i,j]:.2f}", ha="center", va="center", color="w", fontsize=9)
fig.colorbar(im, label="Sharpe"); plt.tight_layout(); plt.show()

## 7 — Regime breakdown
Where does the P&L actually live? Conditioned on the market's (SPY) trend and volatility state.

In [ ]:
strat_ret = A.run_returns(reg, h)
spy = Panel.from_bars(bars).close["SPY"].pct_change().reindex(strat_ret.index)
regime.regime_breakdown(strat_ret, spy).round(3)

## 8 — Tearsheet of the sweep's best cell (with deflation)
Same one-call tearsheet, but passing `sweep_id` so the scorecard shows Deflated Sharpe & PBO.

In [ ]:
tearsheet(reg, bh, sweep_id=sid);